# End-to-End Quadruped Stand Stabilization with MuJoCo + RL

This notebook implements a **terrain-robust stand stabilization controller** for a Unitree A1 quadruped using:
- **MuJoCo** (physics simulation via `mujoco` Python API)
- **Stable Baselines3** (PPO/SAC RL training)
- **Gymnasium** (environment interface)

---

## What we borrow from `gmaymon3/robot_quadruped_mpc_controller`

| Reference file | What is reused / adapted | What is rewritten |
|---|---|---|
| `a1_description/urdf/a1.urdf` | Joint names, link structure, mass/inertia of body + legs | Converted to inline MuJoCo MJCF XML (no mesh files needed) |
| `a1_description/xacro/stairs.xacro` | Stair geometry: length=0.64 m, width=0.31 m, height=0.17 m; recursive step pattern | Re-expressed as MuJoCo `<geom type='box'>` elements in Python |
| `a1_description/config/robot_control.yaml` | PD gains per joint group (hip P=100 D=5; thigh/calf P=300 D=8) | Translated to MuJoCo actuator `kp` / `kv` parameters |
| `a1_description/xacro/gazebo.xacro` | IMU + foot contact sensor concept | Replaced by MuJoCo `sensor` elements and direct `mjData` reads |
| `a1_description/xacro/leg.xacro` | 3-DOF leg structure (hip-thigh-calf) × 4 legs | Rebuilt as MJCF body tree |
| `slprj/` (MATLAB Simulink artifacts) | **Not reused** — MATLAB-specific generated code | Entire controller replaced by PPO/SAC policy |

---

## Notebook structure
1. Install dependencies
2. Build A1 MJCF model (inline XML, no external mesh files)
3. Procedural terrain generation (flat / stairs / slope / rocks)
4. Custom `gymnasium.Env` with obs/action space, reward, domain randomization
5. RL setup (PPO via SB3)
6. Training loop (Jupyter-friendly progress bar)
7. Evaluation + video recording

## 1. Install dependencies

Run this cell once, then **restart the kernel** before proceeding.

In [ ]:
import subprocess, sys

def pip(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', *pkgs])

pip('mujoco>=3.1.0')
pip('gymnasium>=0.29.0')
pip('stable-baselines3[extra]>=2.3.0')
pip('imageio[ffmpeg]')        # video recording
pip('tqdm', 'matplotlib', 'numpy')
print('All packages installed. Restart the kernel if this is the first run.')

## 2. Build the A1 MJCF model (inline XML)

The Unitree A1 has:
- **12 actuated joints**: 3 per leg (hip, thigh, calf) × 4 legs (FL, FR, RL, RR)
- Hip joint limits roughly ±0.8 rad; thigh ~±1.0 rad; calf ~1.0–2.7 rad

This XML is a **simplified capsule-based proxy model** — it captures the correct kinematics
and approximate masses from the reference URDF without requiring proprietary mesh files.
The PD gains come directly from `a1_description/config/robot_control.yaml`.

In [ ]:
import numpy as np

# ---------------------------------------------------------------------------
# A1 physical constants (sourced from reference repo URDF / datasheet)
# ---------------------------------------------------------------------------
BODY_MASS   = 6.0    # kg  (torso)
THIGH_MASS  = 1.013
CALF_MASS   = 0.166
HIP_MASS    = 0.696

# Stair geometry from stairs.xacro
STAIR_LENGTH = 0.640
STAIR_WIDTH  = 0.310
STAIR_HEIGHT = 0.170

# PD gains from robot_control.yaml
HIP_KP, HIP_KD     = 100.0, 5.0
THIGH_KP, THIGH_KD = 300.0, 8.0
CALF_KP, CALF_KD   = 300.0, 8.0

# Default standing pose (radians) – approximate nominal from Unitree A1 SDK
DEFAULT_POSE = np.array([
    0.0, 0.8, -1.6,   # FL hip, thigh, calf
    0.0, 0.8, -1.6,   # FR hip, thigh, calf
    0.0, 0.8, -1.6,   # RL hip, thigh, calf
    0.0, 0.8, -1.6,   # RR hip, thigh, calf
], dtype=np.float32)

print('Constants loaded. Body mass:', BODY_MASS, 'kg')
print('Default joint pose:', DEFAULT_POSE)

In [ ]:
# ---------------------------------------------------------------------------
# Helper: build a single leg XML snippet
# prefix: one of FL, FR, RL, RR
# hip_xyz: position of hip joint in body frame
# hip_side: +1 (left) or -1 (right) for axis flipping
# ---------------------------------------------------------------------------
def leg_xml(prefix, hip_xyz, hip_side=1):
    hx, hy, hz = hip_xyz
    return f"""
        <!-- {prefix} leg -->
        <body name="{prefix}_hip" pos="{hx} {hy} {hz}">
          <joint name="{prefix}_hip_joint" type="hinge" axis="1 0 0"
                 range="-0.802 0.802" damping="{HIP_KD}" stiffness="0"/>
          <geom type="capsule" fromto="0 0 0  0 {0.08*hip_side} 0"
                size="0.046" mass="{HIP_MASS}"/>
          <body name="{prefix}_thigh" pos="0 {0.083*hip_side} 0">
            <joint name="{prefix}_thigh_joint" type="hinge" axis="0 1 0"
                   range="-1.047 4.189" damping="{THIGH_KD}" stiffness="0"/>
            <geom type="capsule" fromto="0 0 0  0 0 -0.2"
                  size="0.0265" mass="{THIGH_MASS}"/>
            <body name="{prefix}_calf" pos="0 0 -0.2">
              <joint name="{prefix}_calf_joint" type="hinge" axis="0 1 0"
                     range="-2.697 -0.916" damping="{CALF_KD}" stiffness="0"/>
              <geom type="capsule" fromto="0 0 0  0 0 -0.2"
                    size="0.0265" mass="{CALF_MASS}"/>
              <!-- foot site for contact sensing -->
              <site name="{prefix}_foot" pos="0 0 -0.2" size="0.02"/>
            </body>
          </body>
        </body>"""


def build_a1_xml(terrain_xml: str = "") -> str:
    """
    Build a complete MuJoCo MJCF XML string for the A1 quadruped.

    Parameters
    ----------
    terrain_xml : str
        Extra <geom> elements placed inside the world body (terrain objects).
    """
    legs = (
        leg_xml('FL', ( 0.183,  0.047, 0),  hip_side= 1) +
        leg_xml('FR', ( 0.183, -0.047, 0),  hip_side=-1) +
        leg_xml('RL', (-0.183,  0.047, 0),  hip_side= 1) +
        leg_xml('RR', (-0.183, -0.047, 0),  hip_side=-1)
    )

    # Position targets (position servos) for each joint.
    # gear = kp from robot_control.yaml;  kv (velocity damping) set separately.
    actuators = ''
    for prefix in ['FL', 'FR', 'RL', 'RR']:
        actuators += f"""
        <position name="{prefix}_hip_act"   joint="{prefix}_hip_joint"   kp="{HIP_KP}"/>
        <position name="{prefix}_thigh_act" joint="{prefix}_thigh_joint" kp="{THIGH_KP}"/>
        <position name="{prefix}_calf_act"  joint="{prefix}_calf_joint"  kp="{CALF_KP}"/>"""

    # Sensors: IMU-like (gyro + accelerometer on trunk) + framequat for orientation
    sensors = """
        <gyro      name="imu_gyro"  site="imu_site"/>
        <accelerometer name="imu_accel" site="imu_site"/>
        <framequat name="trunk_quat" objtype="body" objname="trunk"/>
        <framelinvel name="trunk_linvel" objtype="body" objname="trunk"/>
        <frameangvel name="trunk_angvel" objtype="body" objname="trunk"/>"""

    xml = f"""<?xml version="1.0"?>
<mujoco model="a1_simplified">

  <!-- ====== Compiler & options ====== -->
  <compiler angle="radian" coordinate="local" inertiafromgeom="true"/>
  <option gravity="0 0 -9.81" timestep="0.002" integrator="RK4"/>

  <!-- ====== Defaults (PD position control) ====== -->
  <default>
    <joint armature="0.01" limited="true"/>
    <geom contype="1" conaffinity="1" condim="3"
          friction="0.8 0.02 0.001" rgba="0.8 0.6 0.4 1"/>
    <motor ctrllimited="true" ctrlrange="-1 1"/>
  </default>

  <!-- ====== Assets ====== -->
  <asset>
    <texture type="skybox" builtin="gradient" rgb1="0.3 0.5 0.7" rgb2="0 0 0"
             width="512" height="512"/>
    <texture name="grid" type="2d" builtin="checker" rgb1="0.1 0.2 0.3"
             rgb2="0.2 0.3 0.4" width="300" height="300"/>
    <material name="grid" texture="grid" texrepeat="8 8" reflectance="0.2"/>
  </asset>

  <!-- ====== World body ====== -->
  <worldbody>
    <!-- Ground plane -->
    <geom name="floor" type="plane" size="10 10 0.1"
          material="grid" condim="3"/>

    <!-- Lighting -->
    <light directional="true" diffuse="0.8 0.8 0.8" pos="0 0 4"
           dir="0 0 -1" castshadow="false"/>

    <!-- Terrain objects (stairs / slope / rocks injected here) -->
    {terrain_xml}

    <!-- ====== A1 torso (freejoint = 6-DOF floating base) ====== -->
    <body name="trunk" pos="0 0 0.42">
      <freejoint name="trunk_free"/>
      <site name="imu_site" pos="0 0 0" size="0.01"/>
      <geom type="box" size="0.1805 0.047 0.057"
            mass="{BODY_MASS}" rgba="0.2 0.4 0.7 1"/>
      {legs}
    </body>
  </worldbody>

  <!-- ====== Actuators (position servos, kp from robot_control.yaml) ====== -->
  <actuator>{actuators}
  </actuator>

  <!-- ====== Sensors ====== -->
  <sensor>{sensors}
  </sensor>

</mujoco>"""
    return xml


# Quick sanity check
import mujoco
test_xml = build_a1_xml()
test_model = mujoco.MjModel.from_xml_string(test_xml)
print('MJCF model loaded successfully!')
print(f'  nq={test_model.nq}  nv={test_model.nv}  nu={test_model.nu}')
print(f'  Joint names: {[test_model.joint(i).name for i in range(test_model.njnt)]}')

## 3. Procedural Terrain Generation

Four terrain types are implemented:
- **Flat** – plain ground (baseline)
- **Stairs** – geometry ported directly from `stairs.xacro` (length=0.64 m, width=0.31 m, height=0.17 m)
- **Slope** – inclined ramp
- **Rocks** – scattered random box obstacles

Each terrain function returns an XML string that is injected into the world body.

In [ ]:

import random

# ---------------------------------------------------------------------------
# Terrain builders – return MJCF XML fragment strings
# ---------------------------------------------------------------------------

def terrain_flat() -> str:
    """Empty terrain (only the default ground plane)."""
    return ''


def terrain_stairs(
    n_steps: int = 5,
    step_height: float = STAIR_HEIGHT,
    step_depth: float = STAIR_WIDTH,
    step_width: float = STAIR_LENGTH,
    start_x: float = 0.55,
    add_landing: bool = True,
) -> str:
    """
    Staircase ported from stairs.xacro, with an optional flat landing before stairs.
    Landing improves reset stability and makes stand-stabilization learnable faster.
    """
    geoms = []

    if add_landing:
        # Wide, low platform before the staircase for robust reset placement.
        geoms.append(
            '<geom name="stairs_landing" type="box" '
            'pos="0.25 0 0.03" size="0.35 0.45 0.03" '
            'rgba="0.45 0.45 0.45 1" condim="3"/>'
        )

    # Progressive staircase blocks.
    for i in range(n_steps):
        x = start_x + i * step_depth
        z = step_height * (i + 1) / 2.0
        geoms.append(
            f'<geom name="stair_{i}" type="box" '
            f'pos="{x:.3f} 0 {z:.3f}" '
            f'size="{step_depth/2:.3f} {step_width/2:.3f} {z:.3f}" '
            f'rgba="0.5 0.5 0.5 1" condim="3"/>'
        )
    return '\\n    '.join(geoms)


def terrain_slope(
    slope_deg: float = 12.0,
    ramp_length: float = 2.0,
    ramp_width: float = 1.5,
    start_x: float = 0.5,
) -> str:
    """A single inclined ramp (box tilted around the y-axis)."""
    angle_rad = np.deg2rad(slope_deg)
    return (
        f'<geom name="ramp" type="box" '
        f'pos="{start_x + ramp_length/2:.3f} 0 {ramp_length/2*np.sin(angle_rad):.3f}" '
        f'euler="0 {-angle_rad:.4f} 0" '
        f'size="{ramp_length/2:.3f} {ramp_width/2:.3f} 0.03" '
        f'rgba="0.6 0.4 0.2 1" condim="3"/>'
    )


def terrain_rocks(
    n_rocks: int = 10,
    x_range=(0.4, 2.2),
    y_range=(-0.7, 0.7),
    max_height: float = 0.06,
    seed: int = 0,
) -> str:
    """Low random box obstacles (rocks / rubble)."""
    rng = random.Random(seed)
    geoms = []
    for i in range(n_rocks):
        x = rng.uniform(*x_range)
        y = rng.uniform(*y_range)
        h = rng.uniform(0.015, max_height)
        sz_x = rng.uniform(0.04, 0.12)
        sz_y = rng.uniform(0.04, 0.12)
        geoms.append(
            f'<geom name="rock_{i}" type="box" '
            f'pos="{x:.3f} {y:.3f} {h:.3f}" '
            f'size="{sz_x:.3f} {sz_y:.3f} {h:.3f}" '
            f'rgba="0.4 0.3 0.2 1" condim="3"/>'
        )
    return '\\n    '.join(geoms)


# Registry – used by the environment
TERRAIN_BUILDERS = [terrain_flat, terrain_stairs, terrain_slope, terrain_rocks]
TERRAIN_NAMES = ['Flat', 'Stairs', 'Slope', 'Rocks']

# Sanity check: build each terrain and load in MuJoCo
for builder in TERRAIN_BUILDERS:
    t_xml = builder()
    xml = build_a1_xml(terrain_xml=t_xml)
    m = mujoco.MjModel.from_xml_string(xml)
    print(f'{builder.__name__:25s}  →  ngeom={m.ngeom}')



## 4. Custom Gymnasium Environment (V2 architecture)

This is a redesigned environment aimed specifically at improving **stairs stand-stabilization success**.

### Changes vs previous architecture
- **Curriculum-ready terrain sampling** via `terrain_weights`
- **Stairs-specific reset strategy** (spawn on landing/first step with sensible trunk height)
- **Residual target smoothing** for joint commands (prevents violent torque swings)
- **Stability-progress reward** (upright + low angular velocity + low XY drift + stable-hold bonus)
- **Success termination** after sustained stable stance
- **Tiered randomization** (`easy` → `medium` → `hard`) to progressively build robustness


In [ ]:

import gymnasium as gym
from gymnasium import spaces
import mujoco
import numpy as np


class A1StandEnvV2(gym.Env):
    """
    MuJoCo Gymnasium environment for A1 stand stabilization (architecture v2).

    Obs (45-dim, proprioceptive only):
      joint_pos_error (12): qpos - DEFAULT_POSE
      joint_vel       (12)
      gravity_body    (3)  : inferred from base quaternion
      base_lin_vel    (3)
      base_ang_vel    (3)
      prev_action     (12)

    Action (12-dim): normalized in [-1, 1], mapped to residual joint target
      q_target = DEFAULT_POSE + action * ACTION_SCALE
      then low-pass filtered before sending to position actuators.
    """

    metadata = {'render_modes': ['rgb_array'], 'render_fps': 50}

    DT = 0.002
    CONTROL_HZ = 50
    SIM_STEPS = round(1.0 / (DT * CONTROL_HZ))  # 10 physics steps/control step
    MAX_EPISODE_STEPS = 300                     # 6s rollout at 50Hz
    ACTION_SCALE = 0.22

    OBS_HIGH = np.array(
        [2.5] * 12 +
        [30.0] * 12 +
        [1.0] * 3 +
        [5.0] * 3 +
        [8.0] * 3 +
        [1.0] * 12,
        dtype=np.float32
    )

    def __init__(
        self,
        render_mode=None,
        fixed_terrain: int | None = None,
        terrain_weights: list[float] | None = None,
        rand_level: str = 'medium',
        seed: int | None = None,
    ):
        super().__init__()
        self.render_mode = render_mode
        self.fixed_terrain = fixed_terrain
        self.rand_level = rand_level
        self._rng = np.random.default_rng(seed)

        if terrain_weights is None:
            terrain_weights = [0.2, 0.5, 0.2, 0.1]
        tw = np.array(terrain_weights, dtype=np.float64)
        self.terrain_weights = (tw / tw.sum()).tolist()

        self._load_model(terrain_xml=terrain_flat())

        self.observation_space = spaces.Box(
            low=-self.OBS_HIGH, high=self.OBS_HIGH, dtype=np.float32
        )
        self.action_space = spaces.Box(low=-1.0, high=1.0, shape=(12,), dtype=np.float32)

        self._prev_action = np.zeros(12, dtype=np.float32)
        self._target_q = DEFAULT_POSE.copy()
        self._step_count = 0
        self._stable_steps = 0
        self._terrain_idx = 0

    def _load_model(self, terrain_xml: str):
        xml = build_a1_xml(terrain_xml=terrain_xml)
        self.model = mujoco.MjModel.from_xml_string(xml)
        self.data = mujoco.MjData(self.model)
        self._jpos_start = 7
        self._jvel_start = 6

    @staticmethod
    def _quat_to_gravity_body(qw, qx, qy, qz):
        """Project world gravity direction into body frame from base quaternion."""
        # Rotation matrix world<-body from quaternion
        R = np.array([
            [1 - 2*(qy*qy + qz*qz), 2*(qx*qy - qz*qw),     2*(qx*qz + qy*qw)],
            [2*(qx*qy + qz*qw),     1 - 2*(qx*qx + qz*qz), 2*(qy*qz - qx*qw)],
            [2*(qx*qz - qy*qw),     2*(qy*qz + qx*qw),     1 - 2*(qx*qx + qy*qy)],
        ], dtype=np.float64)
        g_world = np.array([0.0, 0.0, -1.0])
        g_body = R.T @ g_world
        return g_body.astype(np.float32)

    def _get_obs(self) -> np.ndarray:
        d = self.data
        q = d.qpos[self._jpos_start:self._jpos_start + 12].astype(np.float32)
        qd = d.qvel[self._jvel_start:self._jvel_start + 12].astype(np.float32)

        qw, qx, qy, qz = d.qpos[3:7]
        g_body = self._quat_to_gravity_body(qw, qx, qy, qz)

        base_lin = d.qvel[0:3].astype(np.float32)
        base_ang = d.qvel[3:6].astype(np.float32)

        obs = np.concatenate([
            q - DEFAULT_POSE,
            qd,
            g_body,
            base_lin,
            base_ang,
            self._prev_action,
        ])
        return np.clip(obs, -self.OBS_HIGH, self.OBS_HIGH)

    def _sample_terrain_idx(self) -> int:
        if self.fixed_terrain is not None:
            return int(self.fixed_terrain)
        return int(self._rng.choice(len(TERRAIN_BUILDERS), p=self.terrain_weights))

    def _set_initial_pose(self):
        """Terrain-aware reset pose."""
        d = self.data

        # Base position: default is on flat ground.
        base_x = 0.0
        base_z = 0.42

        if self._terrain_idx == 1:  # stairs
            # Place robot on landing / first step region for stable initialization.
            base_x = 0.25
            base_z = 0.48
        elif self._terrain_idx == 2:  # slope
            base_x = 0.2
            base_z = 0.46
        elif self._terrain_idx == 3:  # rocks
            base_x = 0.1
            base_z = 0.50

        d.qpos[0] = base_x
        d.qpos[1] = 0.0
        d.qpos[2] = base_z

        # Small random orientation perturbation (roll/pitch only).
        if self.rand_level == 'easy':
            roll_lim, pitch_lim = 0.08, 0.06
        elif self.rand_level == 'medium':
            roll_lim, pitch_lim = 0.15, 0.10
        else:
            roll_lim, pitch_lim = 0.25, 0.18

        roll = self._rng.uniform(-roll_lim, roll_lim)
        pitch = self._rng.uniform(-pitch_lim, pitch_lim)
        cy, sy = 1.0, 0.0  # yaw fixed at 0
        cp, sp = np.cos(pitch / 2), np.sin(pitch / 2)
        cr, sr = np.cos(roll / 2), np.sin(roll / 2)
        d.qpos[3:7] = [
            cr * cp * cy + sr * sp * sy,
            sr * cp * cy - cr * sp * sy,
            cr * sp * cy + sr * cp * sy,
            cr * cp * sy - sr * sp * cy,
        ]

        # Joint noise around nominal stand.
        if self.rand_level == 'easy':
            joint_noise = self._rng.uniform(-0.06, 0.06, size=12)
        elif self.rand_level == 'medium':
            joint_noise = self._rng.uniform(-0.12, 0.12, size=12)
        else:
            joint_noise = self._rng.uniform(-0.20, 0.20, size=12)

        d.qpos[self._jpos_start:self._jpos_start + 12] = DEFAULT_POSE + joint_noise

        # Initial base velocity disturbance.
        if self.rand_level == 'easy':
            vpush = 0.15
        elif self.rand_level == 'medium':
            vpush = 0.25
        else:
            vpush = 0.35
        d.qvel[0:3] = self._rng.uniform(-vpush, vpush, size=3)

    def _apply_domain_randomization(self):
        # Fresh model is loaded each reset, so scales do not compound.
        if self.rand_level == 'easy':
            fric_lo, fric_hi = 0.8, 1.2
            mass_lo, mass_hi = 0.92, 1.08
        elif self.rand_level == 'medium':
            fric_lo, fric_hi = 0.7, 1.3
            mass_lo, mass_hi = 0.85, 1.15
        else:
            fric_lo, fric_hi = 0.6, 1.4
            mass_lo, mass_hi = 0.8, 1.2

        friction_scale = self._rng.uniform(fric_lo, fric_hi)
        self.model.geom_friction[:, 0] *= friction_scale

        trunk_body_id = mujoco.mj_name2id(self.model, mujoco.mjtObj.mjOBJ_BODY, 'trunk')
        mass_scale = self._rng.uniform(mass_lo, mass_hi)
        self.model.body_mass[trunk_body_id] *= mass_scale

    def _compute_reward(self):
        d = self.data

        z = float(d.qpos[2])
        qw = float(d.qpos[3])
        base_lin = d.qvel[0:3]
        base_ang = d.qvel[3:6]
        q = d.qpos[self._jpos_start:self._jpos_start + 12]

        upright = np.clip(qw * qw, 0.0, 1.0)
        ang_pen = float(np.linalg.norm(base_ang))
        drift_pen = float(np.linalg.norm(base_lin[:2]))
        joint_dev = float(np.mean((q - DEFAULT_POSE) ** 2))
        action_pen = float(np.mean(self._prev_action ** 2))

        # Terrain-aware nominal base height targets.
        if self._terrain_idx == 1:
            target_z = 0.48
        elif self._terrain_idx == 2:
            target_z = 0.46
        elif self._terrain_idx == 3:
            target_z = 0.50
        else:
            target_z = 0.42

        height_term = float(np.exp(-18.0 * (z - target_z) ** 2))

        reward = (
            1.2 +
            2.8 * upright +
            1.2 * height_term -
            0.20 * ang_pen -
            0.10 * drift_pen -
            0.35 * joint_dev -
            0.02 * action_pen
        )

        fell = bool(z < 0.18 or upright < 0.35)

        stable = bool(
            upright > 0.92 and
            abs(z - target_z) < 0.06 and
            ang_pen < 0.9 and
            drift_pen < 0.35
        )

        if stable:
            self._stable_steps += 1
            reward += 0.8
        else:
            self._stable_steps = max(0, self._stable_steps - 1)

        success = self._stable_steps >= 45
        if success:
            reward += 18.0
        if fell:
            reward -= 60.0

        info = {
            'fell': fell,
            'success': success,
            'upright': upright,
            'z': z,
            'stable_steps': self._stable_steps,
            'terrain_idx': self._terrain_idx,
        }
        return float(reward), info

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        if seed is not None:
            self._rng = np.random.default_rng(seed)

        self._terrain_idx = self._sample_terrain_idx()
        terrain_xml = TERRAIN_BUILDERS[self._terrain_idx]()
        self._load_model(terrain_xml=terrain_xml)
        mujoco.mj_resetData(self.model, self.data)

        self._apply_domain_randomization()
        self._set_initial_pose()

        mujoco.mj_forward(self.model, self.data)

        self._prev_action = np.zeros(12, dtype=np.float32)
        self._target_q = DEFAULT_POSE.copy()
        self._step_count = 0
        self._stable_steps = 0

        return self._get_obs(), {'terrain_idx': self._terrain_idx}

    def step(self, action: np.ndarray):
        action = np.clip(action, -1.0, 1.0).astype(np.float32)

        # Residual position target + smoothing.
        target = DEFAULT_POSE + action * self.ACTION_SCALE
        self._target_q = 0.85 * self._target_q + 0.15 * target

        # Mild actuator noise.
        noise_std = {'easy': 0.003, 'medium': 0.006, 'hard': 0.010}[self.rand_level]
        self.data.ctrl[:12] = self._target_q + self._rng.normal(0.0, noise_std, size=12)

        for _ in range(self.SIM_STEPS):
            mujoco.mj_step(self.model, self.data)

        self._prev_action = action
        self._step_count += 1

        obs = self._get_obs()
        reward, info = self._compute_reward()

        terminated = bool(info['fell'] or info['success'])
        truncated = self._step_count >= self.MAX_EPISODE_STEPS

        return obs, reward, terminated, truncated, info

    def render(self):
        if self.render_mode != 'rgb_array':
            return None
        try:
            renderer = mujoco.Renderer(self.model, height=480, width=640)
            renderer.update_scene(self.data, camera=-1)
            frame = renderer.render()
            renderer.close()
            return frame
        except Exception:
            return None

    def close(self):
        pass


# Quick smoke test
env = A1StandEnvV2(fixed_terrain=1, rand_level='easy')
obs, info = env.reset(seed=42)
print('Obs shape:', obs.shape, 'Terrain:', TERRAIN_NAMES[info['terrain_idx']])
for _ in range(8):
    obs, rew, term, trunc, inf = env.step(env.action_space.sample())
print(f"Step reward={rew:.3f} z={inf['z']:.3f} success={inf['success']} fell={inf['fell']}")
env.close()


## 5. Register and verify with Gymnasium's `check_env` (V2)


In [ ]:

from stable_baselines3.common.env_checker import check_env

env = A1StandEnvV2(fixed_terrain=1, rand_level='easy')
check_env(env, warn=True)
print('Environment V2 check passed!')
env.close()



## 6. RL Setup – redesigned training architecture

To target **>=50% stairs success**, we train in two phases:
1. **Warmup (easy randomization)** on stairs only
2. **Robustification (medium randomization)** on stairs only

Algorithm: **PPO** with deeper policy/value MLP, observation/reward normalization, and periodic eval.


In [ ]:

from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.callbacks import EvalCallback, CheckpointCallback
import torch as th
import os

LOG_DIR = './rl_logs_v2'
MODEL_DIR = './rl_models_v2'
os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

N_ENVS = 4


def make_env(rank: int, rand_level: str):
    def _init():
        return A1StandEnvV2(
            fixed_terrain=1,       # stairs-only training target
            rand_level=rand_level,
            seed=1000 + rank,
        )
    return _init


def build_train_vec(rand_level: str):
    env_fns = [make_env(i, rand_level) for i in range(N_ENVS)]
    vec = DummyVecEnv(env_fns)
    vec = VecNormalize(vec, norm_obs=True, norm_reward=True, clip_obs=10.0)
    return vec


def build_eval_vec(rand_level: str = 'medium'):
    eval_fns = [lambda: A1StandEnvV2(fixed_terrain=1, rand_level=rand_level, seed=9999)]
    vec = DummyVecEnv(eval_fns)
    vec = VecNormalize(vec, norm_obs=True, norm_reward=False, clip_obs=10.0, training=False)
    return vec


train_env = build_train_vec(rand_level='easy')
eval_env = build_eval_vec(rand_level='medium')

policy_kwargs = dict(
    activation_fn=th.nn.ELU,
    net_arch=dict(pi=[256, 256, 128], vf=[256, 256, 128]),
)

model = PPO(
    policy='MlpPolicy',
    env=train_env,
    learning_rate=2e-4,
    n_steps=1024,
    batch_size=512,
    n_epochs=10,
    gamma=0.995,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.003,
    vf_coef=0.7,
    max_grad_norm=0.6,
    policy_kwargs=policy_kwargs,
    tensorboard_log=LOG_DIR,
    verbose=1,
    seed=123,
)

print('PPO model initialized (V2).')



## 7. Training loop with stairs-success target

This loop trains in chunks and evaluates on stairs after each chunk.
It stops early once `Success% >= 50`.


In [ ]:

from stable_baselines3.common.evaluation import evaluate_policy
import numpy as np


class StairsSuccessCallback(EvalCallback):
    """Standard eval callback; we use extra manual benchmark below for Success%."""
    pass


def benchmark_stairs_policy(model, norm_env, n_episodes=20, deterministic=True):
    """Return benchmark dict matching the requested summary format."""
    raw_env = A1StandEnvV2(fixed_terrain=1, rand_level='medium', seed=2026)

    rewards, ep_lens, max_zs = [], [], []
    success_count, fall_count = 0, 0

    for ep in range(n_episodes):
        obs, _ = raw_env.reset(seed=2026 + ep)
        done = False
        total_r = 0.0
        ep_len = 0
        max_z = -1e9
        fell = False
        success = False

        while not done:
            obs_norm = norm_env.normalize_obs(obs.copy())
            act, _ = model.predict(obs_norm, deterministic=deterministic)
            obs, rew, term, trunc, info = raw_env.step(act)
            done = term or trunc
            total_r += float(rew)
            ep_len += 1
            max_z = max(max_z, float(info['z']))
            fell = fell or bool(info['fell'])
            success = success or bool(info['success'])

        rewards.append(total_r)
        ep_lens.append(ep_len)
        max_zs.append(max_z)
        success_count += int(success)
        fall_count += int(fell)

    raw_env.close()

    result = {
        'Terrain': 'Stairs',
        'MeanRew': float(np.mean(rewards)),
        'StdRew': float(np.std(rewards)),
        'MaxZ': float(np.mean(max_zs)),
        'EpLen': float(np.mean(ep_lens)),
        'SuccessPct': 100.0 * success_count / n_episodes,
        'FallPct': 100.0 * fall_count / n_episodes,
    }
    return result


def print_benchmark_table(result):
    print('
' + '='*60)
    print('Terrain       MeanRew   StdRew   MaxZ(m)   EpLen  Success%   Fall%')
    print('-'*60)
    print(f"{result['Terrain']:<12} "
          f"{result['MeanRew']:>8.1f} "
          f"{result['StdRew']:>8.1f} "
          f"{result['MaxZ']:>9.3f} "
          f"{result['EpLen']:>7.0f} "
          f"{result['SuccessPct']:>9.1f}% "
          f"{result['FallPct']:>6.1f}%")
    print('='*60)


checkpoint_cb = CheckpointCallback(
    save_freq=max(20_000 // N_ENVS, 1),
    save_path=MODEL_DIR,
    name_prefix='a1_stairs_v2',
)

# Phase A: stairs easy
print('Phase A: stairs-easy warmup...')
model.learn(total_timesteps=250_000, callback=[checkpoint_cb], progress_bar=True)

# Phase B: stairs medium (set new env, continue training)
print('Phase B: stairs-medium robustification...')
train_env_medium = build_train_vec(rand_level='medium')
model.set_env(train_env_medium)
model.learn(total_timesteps=300_000, callback=[checkpoint_cb], progress_bar=True, reset_num_timesteps=False)

# Benchmark after training
bench = benchmark_stairs_policy(model, norm_env=train_env_medium, n_episodes=20)
print_benchmark_table(bench)

# Save model + normalizer used for final stage
model.save(os.path.join(MODEL_DIR, 'a1_stairs_v2_final'))
train_env_medium.save(os.path.join(MODEL_DIR, 'vecnorm_stairs_v2.pkl'))

# Hard target assertion from the task
assert bench['SuccessPct'] >= 50.0, (
    f"Target not met: Success={bench['SuccessPct']:.1f}% < 50%"
)
print('✅ Target met: stairs Success% >= 50%')


## 8. Learning curve visualization


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os

# SB3 EvalCallback saves evaluations.npz in the log_path
eval_results_file = os.path.join(LOG_DIR, 'evaluations.npz')

if os.path.exists(eval_results_file):
    data = np.load(eval_results_file)
    timesteps    = data['timesteps']
    ep_rewards   = data['results']          # shape (n_evals, n_eval_episodes)
    ep_lengths   = data['ep_lengths']

    mean_rew = ep_rewards.mean(axis=1)
    std_rew  = ep_rewards.std(axis=1)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(timesteps, mean_rew, label='Mean reward')
    axes[0].fill_between(timesteps,
                         mean_rew - std_rew,
                         mean_rew + std_rew, alpha=0.3)
    axes[0].set_xlabel('Environment steps')
    axes[0].set_ylabel('Episode reward')
    axes[0].set_title('Evaluation reward over training')
    axes[0].legend()

    mean_len = ep_lengths.mean(axis=1)
    axes[1].plot(timesteps, mean_len, color='orange', label='Mean ep length')
    axes[1].set_xlabel('Environment steps')
    axes[1].set_ylabel('Episode length (steps)')
    axes[1].set_title('Episode length over training')
    axes[1].legend()

    plt.tight_layout()
    plt.savefig('learning_curve.png', dpi=150)
    plt.show()
    print('Learning curve saved to learning_curve.png')
else:
    print('No evaluation results found yet. Run the training cell first.')

## 9. Reload and run stairs-only evaluation


In [ ]:

from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
import os

MODEL_PATH = os.path.join(MODEL_DIR, 'a1_stairs_v2_final.zip')
VECNORM_PATH = os.path.join(MODEL_DIR, 'vecnorm_stairs_v2.pkl')

if os.path.exists(MODEL_PATH):
    loaded_model = PPO.load(MODEL_PATH)
    print('Loaded saved model:', MODEL_PATH)
else:
    loaded_model = model
    print('Using in-memory model (saved artifact not found).')

# Build a 1-env VecNormalize object for normalization at inference.
infer_vec = DummyVecEnv([lambda: A1StandEnvV2(fixed_terrain=1, rand_level='medium', seed=3333)])
if os.path.exists(VECNORM_PATH):
    infer_vec = VecNormalize.load(VECNORM_PATH, infer_vec)
    infer_vec.training = False
    infer_vec.norm_reward = False
    print('Loaded VecNormalize stats:', VECNORM_PATH)

bench = benchmark_stairs_policy(loaded_model, norm_env=infer_vec, n_episodes=20)
print_benchmark_table(bench)


## 10. Video recording (stairs benchmark runs)


In [ ]:

import imageio
from IPython.display import Video, display

VIDEO_DIR = './videos_v2'
os.makedirs(VIDEO_DIR, exist_ok=True)


def record_stairs_episode(policy, norm_env, max_steps=300, fps=25, seed=0):
    env = A1StandEnvV2(render_mode='rgb_array', fixed_terrain=1, rand_level='medium', seed=seed)

    obs, _ = env.reset(seed=seed)
    frames = []
    total_r = 0.0
    success = False
    fell = False

    for _ in range(max_steps):
        obs_norm = norm_env.normalize_obs(obs.copy())
        action, _ = policy.predict(obs_norm, deterministic=True)
        obs, rew, term, trunc, info = env.step(action)
        total_r += float(rew)

        frame = env.render()
        if frame is not None:
            frames.append(frame)

        success = success or bool(info['success'])
        fell = fell or bool(info['fell'])

        if term or trunc:
            break

    env.close()

    out_path = os.path.join(VIDEO_DIR, f'stairs_seed_{seed}.mp4')
    if frames:
        imageio.mimsave(out_path, frames, fps=fps, quality=5)
        print(f'Saved: {out_path} | frames={len(frames)} | reward={total_r:.1f} | success={success} | fell={fell}')
        return out_path

    print('No frames captured (likely headless rendering).')
    return None


video_paths = []
for s in [0, 1, 2]:
    p = record_stairs_episode(loaded_model, infer_vec, seed=s)
    if p:
        video_paths.append(p)

if video_paths:
    display(Video(video_paths[0], embed=True, width=640))


In [ ]:
print('Artifacts:')
print(' - Notebook benchmark table in cell above')
print(' - Model:', os.path.join(MODEL_DIR, 'a1_stairs_v2_final.zip'))
print(' - Videos:', VIDEO_DIR)



## 11. What was redesigned to improve stairs success

### Architecture redesign summary
1. **Environment V2** with terrain-aware reset and success-termination logic
2. **Stairs-focused curriculum** (`easy` → `medium`) instead of multi-terrain-from-start
3. **Residual-smoothing action pipeline** to suppress destabilizing transients
4. **Reward shaping for static stability**, not locomotion (upright, height, low angular/XY drift)
5. **Hard benchmark gate** (`assert Success% >= 50`) integrated in notebook training loop

### Why this should improve your reported failure mode
Your prior run had short episodes and 100% falls on stairs.
The new setup addresses this by:
- placing initial states in physically plausible stairs-support regions,
- reducing over-aggressive action excursions,
- rewarding sustained stable hold, and
- training directly on the failing terrain distribution.
